## ML_1016: Multi-Head Attention

**Difficulty**: Hard | **TorchCode**: #06

### Problem
Implement Multi-Head Attention with separate Q/K/V/O projections. Split the embedding into `num_heads` parallel heads, run scaled dot-product attention per head, then concatenate and project.

### Formula
$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O, \quad \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$
$$d_k = d_{\text{model}} / h$$

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        B, S_q, _ = Q.shape
        S_k = K.shape[1]
        q = self.W_q(Q).view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(K).view(B, S_k, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(V).view(B, S_k, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        out = attn.transpose(1, 2).contiguous().view(B, S_q, -1)
        return self.W_o(out)

In [ ]:
import torch
import torch.nn as nn
import math

# --- Test 1: Output shape ---
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
assert out.shape == (2, 6, 32)

# --- Test 2: nn.Linear projections with correct shapes ---
for name in ['W_q', 'W_k', 'W_v', 'W_o']:
    layer = getattr(mha, name)
    assert isinstance(layer, nn.Linear)
    assert layer.weight.shape == (32, 32)
    assert layer.weight.requires_grad

# --- Test 3: Numerical correctness vs reference ---
torch.manual_seed(0)
D, H = 16, 2; d_k = D // H
mha2 = MultiHeadAttention(d_model=D, num_heads=H)
Q = torch.randn(1, 4, D); K = torch.randn(1, 4, D); V = torch.randn(1, 4, D)
out = mha2.forward(Q, K, V)
q = mha2.W_q(Q).view(1, 4, H, d_k).transpose(1, 2)
k = mha2.W_k(K).view(1, 4, H, d_k).transpose(1, 2)
v = mha2.W_v(V).view(1, 4, H, d_k).transpose(1, 2)
scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
ref = mha2.W_o(torch.matmul(torch.softmax(scores, dim=-1), v).transpose(1, 2).contiguous().view(1, 4, D))
assert torch.allclose(out, ref, atol=1e-5)

# --- Test 4: Gradient flow ---
x = torch.randn(1, 4, 16, requires_grad=True)
mha2.forward(x, x, x).sum().backward()
assert x.grad is not None

# --- Test 5: Cross-attention (seq_q != seq_k) ---
mha3 = MultiHeadAttention(d_model=32, num_heads=4)
Q2 = torch.randn(1, 3, 32); K2 = torch.randn(1, 7, 32); V2 = torch.randn(1, 7, 32)
assert mha3.forward(Q2, K2, V2).shape == (1, 3, 32)

print('All tests passed!')